<a href="https://colab.research.google.com/github/ajit-ai/QuantumComputing/blob/main/Deutsch_Jozsa_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install qiskit
!pip install qiskit qiskit-aer
!pip install qiskit-aer-gpu
!pip install qiskit-aer-gpu-cu11
!pip install qiskit-aer
!pip install qiskit[visualization] qiskit-aer

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement qiskit-aer-gpu (from versions: none)
ERROR: No matching distribution found for qiskit-aer-gpu


Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement qiskit-aer-gpu-cu11 (from versions: none)
ERROR: No matching distribution found for qiskit-aer-gpu-cu11


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.primitives import Sampler
import numpy as np

# Oracle for a constant function (f(x) = 0 or 1)
def constant_oracle(qc, n, value):
    """
    Adds a constant oracle to the circuit.
    value: 0 or 1 (constant output of f(x))
    """
    if value == 1:
        qc.x(n)  # Flip the output qubit if f(x) = 1

# Oracle for a balanced function (e.g., f(x) = x_i for some i)
def balanced_oracle(qc, n, bit_index):
    """
    Adds a balanced oracle where f(x) = x[bit_index].
    n: number of input qubits
    bit_index: index of the bit to check (0 to n-1)
    """
    qc.cx(bit_index, n)  # CNOT: f(x) = 1 if x[bit_index] = 1

# Deutsch-Jozsa algorithm
def deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    """
    Implements Deutsch-Jozsa algorithm.
    n: number of input qubits
    oracle_type: "constant" or "balanced"
    oracle_param: 0 or 1 for constant, bit_index for balanced
    """
    # Define registers
    qreg = QuantumRegister(n + 1, 'q')  # n input qubits + 1 output qubit
    creg = ClassicalRegister(n, 'c')    # Measure only input qubits
    qc = QuantumCircuit(qreg, creg)

    # Step 1: Prepare initial state
    qc.h(qreg[0:n])       # Hadamard on input qubits
    qc.x(qreg[n])         # Flip output qubit to |1>
    qc.h(qreg[n])         # Hadamard on output qubit

    # Step 2: Apply oracle
    if oracle_type == "constant":
        constant_oracle(qc, n, oracle_param)
    elif oracle_type == "balanced":
        balanced_oracle(qc, n, oracle_param)
    else:
        raise ValueError("oracle_type must be 'constant' or 'balanced'")

    # Step 3: Apply Hadamard gates again to input qubits
    qc.h(qreg[0:n])

    # Step 4: Measure input qubits
    qc.measure(qreg[0:n], creg)

    return qc

# Run and analyze the circuit
def run_deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    qc = deutsch_jozsa(n, oracle_type, oracle_param)

    # Use Sampler primitive with AerSimulator
    simulator = AerSimulator()
    sampler = Sampler()
    job = sampler.run(qc, shots=1024)
    result = job.result()
    counts = result.quasi_dists[0].binary_probabilities()

    # Determine if function is constant or balanced
    # If all 0s (|0...0>), it's constant; otherwise, balanced
    is_constant = '0' * n in counts and len(counts) == 1
    print(f"Measurement counts: {counts}")
    print(f"Function is {'constant' if is_constant else 'balanced'}")

    return counts

# Test the algorithm
if __name__ == "__main__":
    # Test with n = 3 qubits
    n = 3

    # Test 1: Constant function (f(x) = 0)
    print("\nTesting constant function f(x) = 0:")
    run_deutsch_jozsa(n, "constant", 0)

    # Test 2: Constant function (f(x) = 1)
    print("\nTesting constant function f(x) = 1:")
    run_deutsch_jozsa(n, "constant", 1)

    # Test 3: Balanced function (f(x) = x_1)
    print("\nTesting balanced function f(x) = x_1:")
    run_deutsch_jozsa(n, "balanced", 1)


Testing constant function f(x) = 0:
Measurement counts: {'000': 1.0}
Function is constant

Testing constant function f(x) = 1:
Measurement counts: {'000': 1.0}
Function is constant

Testing balanced function f(x) = x_1:
Measurement counts: {'010': 1.0}
Function is balanced


In [6]:
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
import numpy as np

def deutsch_jozsa_algorithm(n, oracle_type="balanced"):
    # Create a quantum circuit with n qubits for input and 1 auxiliary qubit
    qc = QuantumCircuit(n + 1, n)
    
    # Step 1: Initialize the auxiliary qubit to |1>
    qc.x(n)  # Apply X gate to set auxiliary qubit to |1>
    
    # Step 2: Apply Hadamard gates to all qubits
    for qubit in range(n + 1):
        qc.h(qubit)
    
    # Step 3: Apply the oracle
    if oracle_type == "constant":
        # Constant oracle: does nothing (e.g., f(x) = 0 for all x)
        pass
    elif oracle_type == "balanced":
        # Balanced oracle: applies X to auxiliary qubit for half the inputs
        # Example: f(x) = 1 if x has odd number of 1s, else 0
        for qubit in range(n):
            qc.cx(qubit, n)  # CNOT from each input qubit to auxiliary qubit
    
    # Step 4: Apply Hadamard gates to input qubits
    for qubit in range(n):
        qc.h(qubit)
    
    # Step 5: Measure the input qubits
    for qubit in range(n):
        qc.measure(qubit, qubit)
    
    return qc

# Parameters
oracle_type = "balanced"  # Can be "constant" or "balanced"

# Create and run the circuit
circuit = deutsch_jozsa_algorithm(n, oracle_type)
simulator = AerSimulator()
result = simulator.run(circuit, shots=1024).result()
counts = result.get_counts()

# Print results
print(f"Oracle type: {oracle_type}")
print(f"Measurement results: {counts}")

# Optional: Visualize the circuit and results
# circuit.draw(output='mpl').show()  # Uncomment to draw circuit
# plot_histogram(counts).show()  # Uncomment to plot histogram

Oracle type: balanced
Measurement results: {'111': 1024}


In [8]:
# Deutsch-Jozsa Algorithm in Python with Qiskit

from qiskit import QuantumCircuit

def deutsch_jozsa_oracle(qc, n, oracle_type="constant", oracle_param=0):
    # Constant oracle: f(x) = 0 or 1
    if oracle_type == "constant":
        if oracle_param == 1:
            qc.x(n)  # Flip output qubit if f(x) = 1
    # Balanced oracle: f(x) = x[oracle_param]
    elif oracle_type == "balanced":
        qc.cx(oracle_param, n)
    else:
        raise ValueError("oracle_type must be 'constant' or 'balanced'")

def deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    qc = QuantumCircuit(n + 1, n)
    # Step 1: Prepare |0...0>|1>
    for i in range(n):
        qc.h(i)
    qc.x(n)
    qc.h(n)
    # Step 2: Oracle
    deutsch_jozsa_oracle(qc, n, oracle_type, oracle_param)
    # Step 3: Hadamard on input qubits
    for i in range(n):
        qc.h(i)
    # Step 4: Measure input qubits
    for i in range(n):
        qc.measure(i, i)
    return qc

def run_deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    qc = deutsch_jozsa(n, oracle_type, oracle_param)
    job = simulator.run(qc, shots=1024)
    result = job.result()
    counts = result.get_counts()
    print("Measurement counts:", counts)
    if '0'*n in counts and len(counts) == 1:
        print("Function is constant")
    else:
        print("Function is balanced")

# Example usage
if __name__ == "__main__":
    n = 3
    print("Constant oracle (f(x)=0):")
    run_deutsch_jozsa(n, "constant", 0)
    print("\nConstant oracle (f(x)=1):")
    run_deutsch_jozsa(n, "constant", 1)
    print("\nBalanced oracle (f(x)=x1):")
    run_deutsch_jozsa(n, "balanced", 1)

Constant oracle (f(x)=0):
Measurement counts: {'000': 1024}
Function is constant

Constant oracle (f(x)=1):
Measurement counts: {'000': 1024}
Function is constant

Balanced oracle (f(x)=x1):
Measurement counts: {'010': 1024}
Function is balanced


In [10]:
from qiskit import QuantumCircuit

def deutsch_jozsa_oracle(qc, n, oracle_type="constant", oracle_param=0):
    # Constant oracle: f(x) = 0 or 1
    if oracle_type == "constant":
        if oracle_param == 1:
            qc.x(n)  # Flip output qubit if f(x) = 1
    # Balanced oracle: f(x) = x[oracle_param]
    elif oracle_type == "balanced":
        qc.cx(oracle_param, n)
    # Balanced oracle: parity (odd number of 1s)
    elif oracle_type == "parity":
        for i in range(n):
            qc.cx(i, n)
    # Balanced oracle: f(x) = x0 XOR x1
    elif oracle_type == "xor":
        qc.cx(0, n)
        qc.cx(1, n)
    # Balanced oracle: f(x) = x0 AND x1
    elif oracle_type == "and":
        qc.ccx(0, 1, n)
    else:
        raise ValueError("Unknown oracle_type")

def deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    qc = QuantumCircuit(n + 1, n)
    # Prepare |0...0>|1>
    for i in range(n):
        qc.h(i)
    qc.x(n)
    qc.h(n)
    # Oracle
    deutsch_jozsa_oracle(qc, n, oracle_type, oracle_param)
    # Hadamard on input qubits
    for i in range(n):
        qc.h(i)
    # Measure input qubits
    for i in range(n):
        qc.measure(i, i)
    return qc

def run_deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    qc = deutsch_jozsa(n, oracle_type, oracle_param)
    job = simulator.run(qc, shots=1024)
    result = job.result()
    counts = result.get_counts()
    print(f"Oracle: {oracle_type}, param: {oracle_param}")
    print("Measurement counts:", counts)
    if '0'*n in counts and len(counts) == 1:
        print("Function is constant\n")
    else:
        print("Function is balanced\n")

# Five different use cases
n = 3
run_deutsch_jozsa(n, "constant", 0)      # f(x) = 0
run_deutsch_jozsa(n, "constant", 1)      # f(x) = 1
run_deutsch_jozsa(n, "balanced", 1)      # f(x) = x1
run_deutsch_jozsa(n, "parity")           # f(x) = parity of x
run_deutsch_jozsa(n, "xor")              # f(x) = x0 XOR x1

Oracle: constant, param: 0
Measurement counts: {'000': 1024}
Function is constant

Oracle: constant, param: 1
Measurement counts: {'000': 1024}
Function is constant

Oracle: balanced, param: 1
Measurement counts: {'010': 1024}
Function is balanced

Oracle: parity, param: 0
Measurement counts: {'111': 1024}
Function is balanced

Oracle: xor, param: 0
Measurement counts: {'011': 1024}
Function is balanced

